In [3]:
import optuna
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

from sklearn.metrics import (
    f1_score,
    classification_report
)
from xgboost import XGBClassifier

FILE_PATH = Path("customer_churn_clean.csv")
df = pd.read_csv(FILE_PATH)

X = df.drop(
    columns=[
        "churn"
    ]
)
y = df[
    "churn"
    ]

cat_col = [
    "contract_type",
    "internet_service",
    "payment_method",
    "tech_support"
]

num_col = [
    "monthly_charges",
    "tenure_months",
    "total_charges",
    "support_tickets_90d",
    "satisfaction_score",
]


Churn_Count = y.value_counts().sort_index()
no_churn = Churn_Count[0]
churn = Churn_Count[1]

Imbalance_Ratio = no_churn / churn

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.2,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    random_state=42,
    test_size=0.2,
    stratify=y_train_full
)

ohe = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

ohe.fit(
    X_train[
        cat_col
    ]
)

X_train_encoded = ohe.transform(
    X_train[
        cat_col
    ]
)

X_test_encoded = ohe.transform(
    X_test[
        cat_col
    ]
)

X_val_encoded = ohe.transform(
    X_val[
        cat_col
    ]
)

X_train_num = X_train[
    num_col
].copy()

X_test_num = X_test[
    num_col
].copy()

X_val_num = X_val[
    num_col
].copy()

X_train_ready = pd.concat(
    [
    X_train_num,
    X_train_encoded
    ],
    axis=1

)

X_test_ready = pd.concat(
    [
    X_test_num,
    X_test_encoded
    ],
    axis=1

)

X_val_ready = pd.concat(
    [
    X_val_num,
    X_val_encoded
    ],
    axis=1

)

def objective(trail):
    params = {
        "max_depth" : trail.suggest_int(
            "max_depth",
            3,
            6
        ),

        "learning_rate" : trail.suggest_float(
            "learning_rate",
            0.01,
            0.1
        ),

        "n_estimators" : trail.suggest_int(
            "n_estimators",
            200,
            400
        ),

        "scale_pos_weight" : trail.suggest_float(
            "scale_pos_weight",
            max(
                1.0 ,
                Imbalance_Ratio * 0.5
            ),
            Imbalance_Ratio * 1.5
        ),

        "random_state" : 42,
        "eval_metric" : "logloss",
        "early_stopping_rounds" : 30
    }
    model = XGBClassifier(
        **params
    )

    model.fit(
        X_train_ready,
        y_train,
        eval_set=[
            (
                X_val_ready,
                y_val
            )
        ],
        verbose=False
    )

    val_prediction = model.predict(X_val_ready)

    score = f1_score(y_val, val_prediction)

    return score

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

best_settings = study.best_params
best_settings["random_state"] = 42
best_settings["eval_metric"] = "logloss"
best_settings["early_stopping_rounds"] = 42

final_model = XGBClassifier(**best_settings)

final_model.fit(
        X_train_ready,
        y_train,
        eval_set=[
            (
                X_val_ready,
                y_val
            )
        ],
        verbose=False
    )

pred = final_model.predict(
    X_test_ready
)

prob = final_model.predict_proba(
    X_test_ready
)[:, 1]

print("===== CLASSIFICATION REPORT =====")

print(
    classification_report(
        y_test,
        pred,
        target_names=[
            "no churn",
            "churn"
        ]
    )
)

print("\n===== OPTUNA RESULTS =====")

print(
    f"F1_score : {study.best_value : 2f}"
)

print("\n===== BEST PARAMETERS =====")

for setting_name, setting_value in study.best_params.items():
    print(
        f"{setting_name} : {setting_value}"
    )


[I 2026-09-14 13:08:37,503] A new study created in memory with name: no-name-eea0b440-6e5d-49d3-be28-2aad42ee1574
[I 2026-09-14 13:08:38,776] Trial 0 finished with value: 0.6235093696763203 and parameters: {'max_depth': 6, 'learning_rate': 0.01715617301294472, 'n_estimators': 327, 'scale_pos_weight': 1.7130656094368661}. Best is trial 0 with value: 0.6235093696763203.
[I 2026-09-14 13:08:39,590] Trial 1 finished with value: 0.6467153284671533 and parameters: {'max_depth': 3, 'learning_rate': 0.0801004298131423, 'n_estimators': 383, 'scale_pos_weight': 2.662551018693172}. Best is trial 1 with value: 0.6467153284671533.
[I 2026-09-14 13:08:40,648] Trial 2 finished with value: 0.6362229102167183 and parameters: {'max_depth': 6, 'learning_rate': 0.02697400639475113, 'n_estimators': 351, 'scale_pos_weight': 2.264677636978952}. Best is trial 1 with value: 0.6467153284671533.
[I 2026-09-14 13:08:41,379] Trial 3 finished with value: 0.6484434320425209 and parameters: {'max_depth': 4, 'learning

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

    no churn       0.85      0.69      0.76      1303
       churn       0.57      0.77      0.65       697

    accuracy                           0.72      2000
   macro avg       0.71      0.73      0.71      2000
weighted avg       0.75      0.72      0.72      2000


===== OPTUNA RESULTS =====
F1_score :  0.657833

===== BEST PARAMETERS =====
max_depth : 3
learning_rate : 0.09792319889936069
n_estimators : 288
scale_pos_weight : 2.144709833141037
